In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import PIL
import tensorflow as tf
import tensorflow_hub as hub

from tensorflow import keras


In [ ]:
# Check if GPU is available
print("GPU Available: ", tf.config.list_physical_devices('GPU'))
print("Built with CUDA: ", tf.test.is_built_with_cuda())

# Check TensorFlow version
print("TensorFlow version:", tf.__version__)

In [ ]:
dataset_train=tf.keras.preprocessing.image_dataset_from_directory(
    "train",
    shuffle=True,
    image_size=(256,256),
    batch_size=32
)

In [ ]:
class_names=dataset_train.class_names
class_names

In [ ]:
dataset_val=tf.keras.preprocessing.image_dataset_from_directory(
    "val",
    shuffle=True,
    image_size=(256,256),
    batch_size=32
)

In [ ]:
dataset_test=tf.keras.preprocessing.image_dataset_from_directory(
    "test",
    shuffle=True,
    image_size=(256,256),
    batch_size=32
)

In [ ]:
plt.figure(figsize=(10,10))
for image_batch,label_batch in dataset_train.take(1):
    for i in range(12):
        ax=plt.subplot(3,4,i+1)
        plt.imshow(image_batch[i].numpy().astype("uint8"))
        plt.title(class_names[label_batch[i]])
        plt.axis('off')

In [ ]:
dataset_train = dataset_train.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
dataset_val = dataset_val.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
dataset_test = dataset_test.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
resize_and_rescale = tf.keras.Sequential([
  tf.keras.layers.Resizing(256,256),
  tf.keras.layers.Rescaling(1./255),
])

In [ ]:
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip("horizontal_and_vertical"),
  tf.keras.layers.RandomRotation(0.2),
])

In [ ]:
BATCH_SIZE = 32
IMAGE_SIZE = 256
CHANNELS=3
EPOCHS=50

In [ ]:
dataset_train = dataset_train.map(
    lambda x, y: (data_augmentation(x, training=True), y)
).prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
from keras import models
from keras import layers

In [ ]:
# Create base model using Keras applications
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(256, 256, 3),
    include_top=False,  # Remove the top classification layer
    weights='imagenet',
    pooling='avg'  # Add global average pooling
)
base_model.trainable = False

# Now use this in your Sequential model
n_classes=2
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(256, 256, 3)),
    resize_and_rescale,
    base_model,
    tf.keras.layers.Dense(n_classes, activation='softmax'),
])

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # Just the string name
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    dataset_train,
    batch_size=BATCH_SIZE,
    validation_data=dataset_val,
    verbose=1,
    epochs=5,
)

In [ ]:
scores = model.evaluate(dataset_test)

In [ ]:
for images_batch, labels_batch in dataset_test.take(1):
    
    first_image = images_batch[0].numpy().astype('uint8')
    first_label = labels_batch[0].numpy()
    
    print("first image to predict")
    plt.imshow(first_image)
    print("actual label:",class_names[first_label])
    
    batch_prediction = model.predict(images_batch)
    print("predicted label:",class_names[np.argmax(batch_prediction[0])])

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

In [ ]:
def predict(model, img):
    img_array = tf.keras.preprocessing.image.img_to_array(images[i].numpy())
    img_array = tf.expand_dims(img_array, 0)

    predictions = model.predict(img_array)

    predicted_class = class_names[np.argmax(predictions[0])]
    confidence = round(100 * (np.max(predictions[0])), 2)
    return predicted_class, confidence

In [ ]:
plt.figure(figsize=(15, 15))
for images, labels in dataset_test.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        
        predicted_class, confidence = predict(model, images[i].numpy())
        actual_class = class_names[labels[i]] 
        
        plt.title(f"Actual: {actual_class},\n Predicted: {predicted_class}.\n Confidence: {confidence}%")
        
        plt.axis("off")

In [ ]:
EPOCHS=5
plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(range(EPOCHS), acc, label='Training Accuracy')
plt.plot(range(EPOCHS), val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(range(EPOCHS), loss, label='Training Loss')
plt.plot(range(EPOCHS), val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# Function to compute Grad-CAM
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# Function to display Grad-CAM overlay
def display_gradcam(image, heatmap, alpha=0.4):
    img = tf.keras.preprocessing.image.array_to_img(image)
    heatmap = np.uint8(255 * heatmap)
    jet = plt.cm.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]
    jet_heatmap = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((img.size[0], img.size[1]))
    superimposed_img = tf.keras.preprocessing.image.array_to_img(
        np.array(img) * (1 - alpha) + np.array(jet_heatmap) * alpha
    )
    plt.imshow(superimposed_img)
    plt.axis('off')
    plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import numpy as np

# Predict on test dataset
y_true = []
y_pred = []
for batch, labels in dataset_test:
    preds = model.predict(batch)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

# Print classification report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

# ROC-AUC (Binary)
print(class_names)  # Output should be something like ['NORMAL', 'PNEUMONIA']

# If PNEUMONIA is class 1:
y_prob = []
y_true = []

for batch, labels in dataset_test:
    probs = model.predict(batch)
    y_prob.extend(probs[:, 1])  # class 1 (PNEUMONIA)
    y_true.extend(labels.numpy())

from sklearn.metrics import roc_auc_score
roc_auc = roc_auc_score(y_true, y_prob)
print(f"ROC AUC: {roc_auc:.4f}")


In [ ]:
# The modern, recommended way
model.save('chest_xray_model.keras')

print("Model saved successfully in the new Keras format!")